# Study 942 — The Inverse Tax 🔻

**Is an inverse ETF really a worse short than shorting the index yourself?**

The standard warning on SH, PSQ and SDS levies three charges: you pay an *expense ratio* a
direct shorter does not, you eat the *daily-reset path drag*, and you hand the sponsor the
*financing* a short seller would collect as a rebate. So — the story goes — anyone who can
borrow the shares should short the index outright.

We build the honest alternative and race it: a directly-short book rebalanced daily to the
same −k× NAV, which **owes the index's dividends**, pays a stock-loan fee, and collects
whatever rebate its broker credits (modelled off `^IRX`). Real tape 2007-05-31 → 2026-06-30
(4,799 days), daily total-return closes plus price-only SPY/QQQ closes for the
dividend leg.

*Numbers below are the frozen headline run (`docs/results.md`, Fingerprint `b9485964fb9d`); the
only live cells run the fast offline synthetic control. As-of 2026-06-30.*


## 1. The three charges, and what they are worth

Before any arithmetic, notice what the story leaves out. The ProShares inverse S&P 500 and Nasdaq-100 funds track the **price** index — so their holder never owes the dividends a share-short is charged for. And the funds hold their whole NAV in collateral and swaps struck against a short rate, while a *retail* short seller is usually paid nothing at all on the proceeds. Two credits the folklore never counts, against three charges it does.

## 2. The result — the fund wins, at the financing you actually get

Base case: your own cash earns the bill rate, the short proceeds earn nothing (the ordinary retail arrangement), 30 bps to borrow the shares, 1 bp a day of rebalancing cost on the self-managed book.

In [1]:
R = {'start': '2007-05-31', 'end': '2026-06-30', 'n_days': 4799, 'fp': 'b9485964fb9d', 'credit': 1.0, 'borrow_bps': 30.0, 'cost_bps': 1.0, 'irx_ann': 1.47, 'bil_ann': 1.36, 'irx_bil_diff': 0.11, 'sh_gap': 1.06, 'sh_t': 5.1, 'sh_t_iid': 2.1, 'sh_sd': 13.8, 'sh_ci_lo': 0.71, 'sh_ci_hi': 1.45, 'psq_gap': 0.58, 'psq_t': 2.5, 'psq_sd': 16.6, 'psq_ci_lo': 0.2, 'psq_ci_hi': 0.97, 'sds_gap': 1.19, 'sds_t': 3.78, 'sds_sd': 21.6, 'sds_ci_lo': 0.69, 'sds_ci_hi': 1.74, 'sh_sm': -0.79, 'sh_sm_t': -2.76, 'psq_sm': -0.23, 'psq_sm_t': -0.94, 'sds_sm': -2.51, 'sds_sm_t': -4.87, 'be_sm_sh': 0.46, 'be_sm_psq': 0.84, 'be_sm_sds': 0.14, 'sm_zirp': -2.52, 'sm_zirp_t': -6.61, 'sm_norm': 1.59, 'sm_norm_t': 4.2, 'hac_profile': ((0, 2.1), (1, 2.82), (2, 3.21), (5, 4.61), (9, 5.1), (21, 5.5), (63, 4.34), (126, 3.27), (252, 2.44)), 'gap_acf1': -0.44, 'sh_exsharpe': -0.574, 'dir_exsharpe': -0.627, 'sharpe_diff': 0.052, 'sh_cagr': -11.22, 'dir_cagr': -12.16, 'sh_vol': 19.7, 'dir_vol': 19.8, 'sh_dd': -94.7, 'dir_dd': -95.4, 'sh_raw': 2.18, 'sh_div': 1.85, 'sh_fin': 1.74, 'sh_er': 0.89, 'sh_resid': -0.52, 'sh_beta': 0.9914, 'sh_gamma': 1.187, 'sh_gamma_px': 1.4, 'sh_alpha': 0.33, 'psq_raw': 1.69, 'psq_div': 0.81, 'psq_fin': 2.01, 'psq_er': 0.95, 'psq_resid': -0.18, 'psq_beta': 0.9917, 'psq_gamma': 1.371, 'psq_gamma_px': 1.263, 'sds_raw': 3.39, 'sds_div': 3.7, 'sds_fin': 2.8, 'sds_er': 0.89, 'sds_resid': -2.21, 'sds_beta': 0.9823, 'sds_gamma': 0.954, 'sds_gamma_px': 1.164, 'spy_yield': 1.85, 'qqq_yield': 0.81, 'be_sh': 1.72, 'be_psq': 1.39, 'be_sds': 1.41, 'era_e_n': 2163, 'era_e_gap': 0.16, 'era_e_t': 0.39, 'era_e_rf': 0.49, 'era_l_n': 2636, 'era_l_gap': 1.79, 'era_l_t': 9.54, 'era_l_rf': 2.26, 'zirp_n': 2785, 'zirp_gap': -0.46, 'zirp_t': -1.83, 'zirp_rf': 0.15, 'norm_n': 2014, 'norm_gap': 3.15, 'norm_t': 11.37, 'norm_rf': 3.29, 'c0_b0': 2.22, 'c0_b0_t': 8.74, 'c1_b30': 1.06, 'c1_b30_t': 5.1, 'c15_b0': 0.02, 'c15_b0_t': 0.09, 'c2_b0': -0.72, 'c2_b0_t': -3.81, 'c2_b30': -0.41, 'c2_b30_t': -2.19, 'cost0': 1.01, 'cost0_t': 4.9, 'cost10': 1.42, 'cost10_t': 6.82, 'sh_drag_63': -0.34, 'sh_drag_63_med': -0.26, 'sh_wrap_63': 0.27, 'sds_drag_63': -1.12, 'sds_drag_63_med': -0.76, 'sds_wrap_63': 0.38, 'sds_drag_252': -2.68, 'sh_drag_252': -0.8, 'syn_plant': -2.77, 'syn_plant_t': -22.06, 'syn_null': 0.23, 'syn_null_t': 1.84, 'syn_null_mean': 0.027, 'syn_null_sd': 0.159, 'syn_null_fire': 0, 'syn_plant_mean': -2.973, 'syn_seeds': 12}
print('gap = inverse ETF minus an honest direct short (positive = the FUND wins)')
print(f"  SH  (-1x S&P 500) : {R['sh_gap']:+.2f}%/yr   HAC t {R['sh_t']:+.2f}")
print(f"  PSQ (-1x Nasdaq)  : {R['psq_gap']:+.2f}%/yr   HAC t {R['psq_t']:+.2f}")
print(f"  SDS (-2x S&P 500) : {R['sds_gap']:+.2f}%/yr   HAC t {R['sds_t']:+.2f}")
print(f"  SH bootstrap 95% CI: [{R['sh_ci_lo']:+.2f}, {R['sh_ci_hi']:+.2f}] "
      f"-- clear of zero")

gap = inverse ETF minus an honest direct short (positive = the FUND wins)
  SH  (-1x S&P 500) : +1.06%/yr   HAC t +5.10
  PSQ (-1x Nasdaq)  : +0.58%/yr   HAC t +2.50
  SDS (-2x S&P 500) : +1.19%/yr   HAC t +3.78
  SH bootstrap 95% CI: [+0.71, +1.45] -- clear of zero


The folklore says that column should be **negative**. On all three funds it is positive, and on SH it is positive with a *t* of **5.10**.

> 🔬 **For the quants:** the *t* is Newey-West on the daily return difference (the plain i.i.d. *t* is +2.10; the gap's lag-1 autocorrelation is -0.44, which is why HAC *raises* it), and the interval is a circular block bootstrap (2,000 draws, 21-day blocks) on the annualised mean gap. Both arms are raced excess-of-cash against BIL: excess Sharpe -0.574 (fund) vs -0.627 (direct).

## 2b. The catch — that race is not quite apples to apples

SH tracks the **price** index. Someone who shorts SPY shares is short the **total return** and has to hand over the dividends. So part of what we just measured is not the fund being *better* — it is the fund being short a *different, slightly easier* thing. Charge the fund those dividends too and run the race again:

In [2]:
print('same-mandate race: BOTH arms charged the index dividends')
for tag, head, sm, t in [
    ('SH ', R['sh_gap'],  R['sh_sm'],  R['sh_sm_t']),
    ('PSQ', R['psq_gap'], R['psq_sm'], R['psq_sm_t']),
    ('SDS', R['sds_gap'], R['sds_sm'], R['sds_sm_t']),
]:
    print(f"  {tag}: headline {head:+.2f}%/yr  ->  same-mandate {sm:+.2f}%/yr "
          f"(HAC t {t:+.2f})")
print('\nthe sign flips. both signs are significant. that is the whole study.')

same-mandate race: BOTH arms charged the index dividends
  SH : headline +1.06%/yr  ->  same-mandate -0.79%/yr (HAC t -2.76)
  PSQ: headline +0.58%/yr  ->  same-mandate -0.23%/yr (HAC t -0.94)
  SDS: headline +1.19%/yr  ->  same-mandate -2.51%/yr (HAC t -4.87)

the sign flips. both signs are significant. that is the whole study.


**Both framings are legitimate, and they disagree.** If your question is *"buy the fund or short the shares?"*, the dividends are part of the answer and the fund wins. If your question is *"is the wrapper an efficient way to carry a given short exposure?"*, you take the dividends out and the fund loses — -0.79 %/yr on SH, -2.51 on the −2×. This is why the Signal stamp is **Mixed** rather than a clean yes or no.

## 3. Where the money is — the four terms

Strip the direct book of *all* interest and the raw gap breaks into four pieces. The expense ratio is the only one the folklore names, and it is the smallest.

In [3]:
print(f"SH, raw gap {R['sh_raw']:+.2f}%/yr, against a direct short "
      f"paid no interest at all:")
for label, value in [
    ('dividends a share-short owes, the fund does not', R['sh_div']),
    ('short-rate interest the fund credits on NAV',     R['sh_fin']),
    ('the expense ratio',                               -R['sh_er']),
    ('the wrapper residual (swap spread, slippage)',    R['sh_resid']),
]:
    print(f"   {value:+6.2f}%/yr  {label}")

SH, raw gap +2.18%/yr, against a direct short paid no interest at all:
    +1.85%/yr  dividends a share-short owes, the fund does not
    +1.74%/yr  short-rate interest the fund credits on NAV
    -0.89%/yr  the expense ratio
    -0.52%/yr  the wrapper residual (swap spread, slippage)


SPY paid **1.85%/yr** of dividends over this window — a bill a share-short foots and an SH holder does not. And the fund credits about **1.19 units** of the 13-week bill rate on its NAV, where a retail shorter collects one. Together those two dwarf the 0.89% expense ratio and the 0.52% the wrapper genuinely costs.

## 4. The daily reset is real — and it is not the fund's fault

Over a quarter, keeping a book at −2× costs **-1.12 pp** versus putting a short on once and leaving it (median -0.76); at −1× it costs -0.34 pp. That drag is genuine. But it belongs to the *discipline of holding constant leverage*, not to the wrapper: a self-managed book rebalanced to −1× every day pays exactly the same bill. Once you compare the fund with a **daily-reset** book instead of a static one, the wrapper's own column is small and, if anything, slightly positive (+0.38 pp per quarter on SDS). Blaming the reset on the ETF is a category error.

## 5. So when *is* the folklore right? When money is free.

Cut the sample by the prevailing 13-week bill rate — a level you know at the start of each day, so this is not hindsight.

In [4]:
print(f"bills below 1% (n={R['zirp_n']:,}, avg {R['zirp_rf']:.2f}%/yr): "
      f"gap {R['zirp_gap']:+.2f}%/yr   HAC t {R['zirp_t']:+.2f}")
print(f"bills above 1% (n={R['norm_n']:,}, avg {R['norm_rf']:.2f}%/yr): "
      f"gap {R['norm_gap']:+.2f}%/yr   HAC t {R['norm_t']:+.2f}")
print()
print(f"break-even: the fund wins until your broker credits you {R['be_sh']:.2f} "
      f"units of the bill rate.")
print('retail is about 1.0; a prime-brokerage account is about 2.0.')

bills below 1% (n=2,785, avg 0.15%/yr): gap -0.46%/yr   HAC t -1.83
bills above 1% (n=2,014, avg 3.29%/yr): gap +3.15%/yr   HAC t +11.37

break-even: the fund wins until your broker credits you 1.72 units of the bill rate.
retail is about 1.0; a prime-brokerage account is about 2.0.


In a zero-rate world there is no financing to pass through, so the expense ratio and the reset residual are all that is left and the fund is indeed the marginally worse short — **-0.46%/yr** on the product-vs-product race (*t* = -1.83, short of the bar) and **-2.52%/yr** on the same-mandate one (*t* = -6.61, well past it). When bills pay 3.3% the fund beats a retail direct short by **+3.15%/yr**. The "inverse tax" is a ZIRP-era memory.

## 6. Live check — the machinery is unbiased (offline synthetic, no real data)

We simulate an inverse fund that really *does* leak a known 3%/yr, and a second one that is a costless replicate. The harness must find the first and stay quiet on the second.

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from inverse_tax import data, strategy as st
taxed, t1 = data.synthetic_daily(signal_strength=1.0, seed=942)
fair,  t0 = data.synthetic_daily(signal_strength=0.0, seed=942)
d1, d0 = st.synthetic_detect(taxed), st.synthetic_detect(fair)
print('planted a %+.2f%%/yr tax -> measured %+.2f%%/yr (t %+.2f)'
      % (t1['expected_gap_ann']*100, d1['gap_ann_pct'], d1['gap_t']))
print('planted no tax at all   -> measured %+.2f%%/yr (t %+.2f)'
      % (d0['gap_ann_pct'], d0['gap_t']))

planted a -3.00%/yr tax -> measured -2.77%/yr (t -22.06)
planted no tax at all   -> measured +0.23%/yr (t +1.84)


## Verdict

- **Signal — Mixed.** There is a real, tightly measured structural gap (**+1.06 %/yr** on SH, HAC *t* = **+5.10**, replicated on PSQ and SDS) — but it is a *subsidy*, not the advertised tax, at the financing an ordinary shorter actually gets. Its sign is decided by three things outside the price tape: the level of short rates, what your broker pays you, and whether you charge the fund for dividends it does not owe (do that and SH reads -0.79 %/yr, *t* -2.76).
- **Tradability — Fragile.** Worth about a point a year at retail terms, +3.15 %/yr at 2023-2026 rates — but it is an implementation choice on a hedge, not a return stream (both arms lost 11-12 %/yr while the index quintupled), and it reverses for a prime broker and evaporates at zero rates.
- **What was busted.** The expense ratio is the *smallest* of the three charges; the daily-reset drag is not the wrapper's; and the financing leg — the one that carries the claim — points the other way for everyone who is not a prime broker.